In [1]:
import pandas as pd

# Load the cricket data from the Excel file
df_cricket = pd.read_excel('cricket_data.xlsx')

# Display the first 5 rows and basic info to understand the data structure
print("First 5 rows of the data:")
display(df_cricket.head())

print("\nInformation about the DataFrame:")
df_cricket.info()

First 5 rows of the data:


,Match_ID,Match_Date,Venue,Match_Format,Batsman,Opponent,Runs_Scored,Balls_Faced,Strike_Rate,Wickets_Taken,Overs_Bowled,Runs_Conceded,Match_Won,Year,Month,Batting_StrikeRate,Bowling_Economy,Perf_Score,Player Role
0,M001,2020-01-01,Narendra Modi,T20,Hardik Pandya,England,50.0,183.0,119.38,4,8.7,44,0,2020.0,January,27.32,5.06,150.0,All-Rounder
1,M002,2020-01-06,Chepauk,T20,MS Dhoni,New Zealand,172.0,66.0,163.10,0,5.5,70,1,2020.0,January,260.61,12.73,252.3,Batsman
2,M003,2020-01-11,Arun Jaitley,ODI,MS Dhoni,New Zealand,3.0,NaN,190.81,4,5.3,63,1,2020.0,January,NaN,11.89,103.0,Bowler
3,M004,2020-01-16,Narendra Modi,T20,Virat Kohli,Pakistan,112.0,198.0,189.58,5,6.3,72,1,2020.0,January,56.57,11.43,237.0,All-Rounder
4,M005,2020-01-21,Chinnaswamy,T20,Shubman Gill,Sri Lanka,31.0,83.0,123.12,5,8.4,16,1,2020.0,January,37.35,1.90,156.0,All-Rounder



Information about the DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Match_ID            200 non-null    object 
 1   Match_Date          193 non-null    object 
 2   Venue               200 non-null    object 
 3   Match_Format        200 non-null    object 
 4   Batsman             200 non-null    object 
 5   Opponent            200 non-null    object 
 6   Runs_Scored         187 non-null    float64
 7   Balls_Faced         192 non-null    float64
 8   Strike_Rate         194 non-null    float64
 9   Wickets_Taken       200 non-null    int64  
 10  Overs_Bowled        194 non-null    float64
 11  Runs_Conceded       200 non-null    int64  
 12  Match_Won           200 non-null    int64  
 13  Year                193 non-null    float64
 14  Month               193 non-null    object 
 15  Batting_StrikeRate  179

In [3]:
# Calculate summary statistics for the requested columns
summary_stats = df_cricket[['Runs_Scored', 'Balls_Faced', 'Strike_Rate', 'Bowling_Economy']].describe()
display(summary_stats)

,Runs_Scored,Balls_Faced,Strike_Rate,Bowling_Economy
count,187.000000,192.000000,194.000000,194.000000
mean,82.192513,99.661458,125.557268,11.209485
std,50.162910,55.083135,39.821824,9.861756
min,0.000000,5.000000,60.650000,1.490000
25%,40.000000,50.750000,90.337500,4.670000
50%,78.000000,100.000000,122.780000,8.045000
75%,120.000000,146.250000,159.495000,14.307500
max,179.000000,198.000000,199.560000,51.820000


In [4]:
# Calculate Q1 (25th percentile) and Q3 (75th percentile) for 'Strike_Rate'
Q1_sr = df_cricket['Strike_Rate'].quantile(0.25)
Q3_sr = df_cricket['Strike_Rate'].quantile(0.75)

# Calculate IQR
IQR_sr = Q3_sr - Q1_sr

# Calculate lower and upper bounds for outliers
lower_bound_sr = Q1_sr - 1.5 * IQR_sr
upper_bound_sr = Q3_sr + 1.5 * IQR_sr

# Identify outliers
outliers_sr = df_cricket[(df_cricket['Strike_Rate'] < lower_bound_sr) | (df_cricket['Strike_Rate'] > upper_bound_sr)]

print(f"Q1 for Strike_Rate: {Q1_sr:.2f}")
print(f"Q3 for Strike_Rate: {Q3_sr:.2f}")
print(f"IQR for Strike_Rate: {IQR_sr:.2f}")
print(f"Lower Bound for Outliers: {lower_bound_sr:.2f}")
print(f"Upper Bound for Outliers: {upper_bound_sr:.2f}")

print("\nOutliers in Strike_Rate:")
display(outliers_sr[['Match_ID', 'Batsman', 'Strike_Rate']])

Q1 for Strike_Rate: 90.34
Q3 for Strike_Rate: 159.50
IQR for Strike_Rate: 69.16
Lower Bound for Outliers: -13.40
Upper Bound for Outliers: 263.23

Outliers in Strike_Rate:


,Match_ID,Batsman,Strike_Rate


In [5]:
from scipy.stats import norm
import numpy as np

# Clean 'Strike_Rate' column by dropping NaNs, as they would affect mean/std calculation
strike_rate_clean = df_cricket['Strike_Rate'].dropna()

# Calculate mean and standard deviation of 'Strike_Rate'
mean_sr = np.mean(strike_rate_clean)
std_sr = np.std(strike_rate_clean)

# Calculate the Z-score for a Strike Rate of 120
z_score_120 = (120 - mean_sr) / std_sr

# Calculate the probability P(Strike_Rate > 120)
# The `sf` (survival function) is 1 - cdf, which gives P(X > x)
probability_gt_120 = norm.sf(z_score_120)

print(f"Mean Strike Rate: {mean_sr:.2f}")
print(f"Standard Deviation of Strike Rate: {std_sr:.2f}")
print(f"Z-score for Strike Rate > 120: {z_score_120:.2f}")
print(f"Probability of Strike Rate > 120: {probability_gt_120:.4f}")

Mean Strike Rate: 125.56
Standard Deviation of Strike Rate: 39.72
Z-score for Strike Rate > 120: -0.14
Probability of Strike Rate > 120: 0.5556


In [6]:
from scipy.stats import poisson

# Assumed average number of sixes per match (lambda)
lambda_sixes = 8

print(f"Assumed average number of sixes per match (λ): {lambda_sixes}")
print(f"Expected number of sixes in a match (E[X]): {lambda_sixes}")

# We can also calculate the probability of observing a certain number of sixes, e.g., 10 sixes in a match
prob_10_sixes = poisson.pmf(10, lambda_sixes)
print(f"Probability of observing exactly 10 sixes in a match: {prob_10_sixes:.4f}")

Assumed average number of sixes per match (λ): 8
Expected number of sixes in a match (E[X]): 8
Probability of observing exactly 10 sixes in a match: 0.0993


In [7]:
from scipy.stats import binom

# Number of trials (balls faced)
n = 12
# Number of successes (boundaries hit)
k = 3
# Assumed probability of hitting a boundary on a single ball
p_boundary = 0.25

# Calculate the binomial probability
probability_3_boundaries = binom.pmf(k, n, p_boundary)

print(f"Number of balls (n): {n}")
print(f"Number of boundaries (k): {k}")
print(f"Assumed probability of hitting a boundary per ball (p): {p_boundary}")
print(f"Probability of hitting exactly 3 boundaries out of 12 balls: {probability_3_boundaries:.4f}")

Number of balls (n): 12
Number of boundaries (k): 3
Assumed probability of hitting a boundary per ball (p): 0.25
Probability of hitting exactly 3 boundaries out of 12 balls: 0.2581
